In [ ]:
import tensorflow as tf
import numpy as np
from keras import layers, optimizers, callbacks
from transformers import GPT2TokenizerFast
from keras.saving import register_keras_serializable


# Load text
with open('/content/Data_2.txt','r',encoding='utf-8') as f:
    text = f.read()

seq_len = 100
# ============================================
# GPT-2 Tokenization Setup
# ============================================
print("Initializing GPT-2 tokenizer...")
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')

# Add padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

vocab_size = len(tokenizer)
print(f"Vocabulary size: {vocab_size}")

def encode(text):
    """Encode text using GPT-2 tokenizer"""
    return tokenizer.encode(text, add_special_tokens=False)

def decode(indices):
    """Decode token indices back to text"""
    return tokenizer.decode(indices)
# Encode the entire text
print("Encoding text...")
encoded = encode(text)

In [ ]:

# Encode full text
tokens = tf.constant(encoded, dtype=tf.int32)

# Trim to a clean multiple of (seq_len + 1)
tokens = tokens[: (len(tokens) // (seq_len + 1)) * (seq_len + 1)]

# Reshape into windows
windows = tf.reshape(tokens, (-1, seq_len + 1))

# Dataset before mapping
ds = tf.data.Dataset.from_tensor_slices(windows)

# Input/target split
ds = ds.map(lambda x: (x[:-1], x[1:]),
            num_parallel_calls=tf.data.AUTOTUNE)

# ============================================
# Train / Validation Split
# ============================================

total_sequences = windows.shape[0]
val_size = int(total_sequences * 0.1)   # 10% val
train_size = total_sequences - val_size

train_ds = ds.take(train_size)
val_ds   = ds.skip(train_size)

# ============================================
# Batch + Prefetch
# ============================================

BATCH = 32

train_ds = (
    train_ds
    .shuffle(10000)
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE)
)


In [ ]:
initializer = tf.keras.initializers.TruncatedNormal(stddev=0.02)
ln_eps = 1e-5  # GPT-2 uses small eps

d_model = 512
heads = 8
bias = False
n_layers = 4
dropout = 0.2

class RoPEMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, num_heads, head_dim, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.inner_dim = num_heads * head_dim
        self.dropout_rate = dropout
        self.q_proj = layers.Dense(self.inner_dim, use_bias=bias)
        self.k_proj = layers.Dense(self.inner_dim, use_bias=bias)
        self.v_proj = layers.Dense(self.inner_dim, use_bias=bias)
        self.o_proj = layers.Dense(self.inner_dim, use_bias=bias)
        self.att_dropout = layers.Dropout(dropout)


    # ---------------------- RoPE UTILITIES ----------------------

    def build(self, input_shape):
        # Precompute rotary frequencies
        dim = self.head_dim
        half_dim = dim // 2

        inv_freq = 1.0 / (10000 ** (tf.range(0, half_dim, dtype=tf.float32) / half_dim))
        self.inv_freq = inv_freq  # (D/2,)

    def get_rope_cache(self, T):
        # positions: (T, 1)
        pos = tf.cast(tf.range(T), tf.float32)[:, None]  

        freqs = pos * self.inv_freq[None, :]   # (T, D/2)
        sin = tf.sin(freqs)
        cos = tf.cos(freqs)

        # reshape → (1, 1, T, D/2)
        sin = sin[None, None, :, :]
        cos = cos[None, None, :, :]

        return sin, cos

    def apply_rope(self, x, sin, cos):
            # x: (B, H, T, D)
            B, H, T, D = tf.unstack(tf.shape(x))
            half = D // 2

            x1 = x[:, :, :, :half]
            x2 = x[:, :, :, half:]

            # rotate pairs: (x1, x2) → (x1 * cos - x2 * sin, x2 * cos + x1 * sin)
            roped_x1 = x1 * cos - x2 * sin
            roped_x2 = x2 * cos + x1 * sin

            return tf.concat([roped_x1, roped_x2], axis=-1)


    # ---------------------- HEAD SPLIT/COMBINE ----------------------

    def split_heads(self, x):
        B = tf.shape(x)[0]
        T = tf.shape(x)[1]
        x = tf.reshape(x, (B, T, self.num_heads, self.head_dim))
        return tf.transpose(x, (0, 2, 1, 3))  # (B, H, T, D)

    def combine_heads(self, x):
        x = tf.transpose(x, (0, 2, 1, 3))  # (B, T, H, D)
        B = tf.shape(x)[0]
        T = tf.shape(x)[1]
        return tf.reshape(x, (B, T, self.inner_dim))

    # ---------------------- MAIN FORWARD ----------------------

    def call(self, x):
        B = tf.shape(x)[0]
        T = tf.shape(x)[1]

        q = self.split_heads(self.q_proj(x))
        k = self.split_heads(self.k_proj(x))
        v = self.split_heads(self.v_proj(x))

        # ----- APPLY ROPE -----
        sin, cos = self.get_rope_cache(T)
        q = self.apply_rope(q, sin, cos)
        k = self.apply_rope(k, sin, cos)

        # ----- ATTENTION -----
        att = tf.matmul(q, k, transpose_b=True)
        att = att / tf.math.sqrt(tf.cast(self.head_dim, tf.float32))

        mask = tf.linalg.band_part(tf.ones((T, T)), -1, 0)
        att = tf.where(mask[None, None, :, :] == 0, -1e9, att)

        att = tf.nn.softmax(att, axis=-1)
        att = self.att_dropout(att)

        out = tf.matmul(att, v)
        out = self.combine_heads(out)

        return self.o_proj(out)

def transformer_block(x, d_model, heads, dropout):
    # ---- Attention block (Pre-LN) ----
    ln1 = layers.RMSNormalization(epsilon=ln_eps)(x)
    att = RoPEMultiHeadAttention(
        num_heads=heads,
        head_dim=d_model // heads,
        dropout=dropout
    )(ln1)

    att = layers.Dropout(dropout)(att)
    x = x + att

    # ---- Feed Forward block (Pre-LN) ----
    ln2 = layers.RMSNormalization(epsilon=ln_eps)(x)
    ffn = layers.Dense(d_model * 4, activation='gelu', use_bias=bias,
                       kernel_initializer=initializer)(ln2)
    #ffn = layers.Dropout(dropout)(ffn)
    ffn = layers.Dense(d_model, use_bias=bias, kernel_initializer=initializer)(ffn)
    #ffn = layers.Dropout(dropout)(ffn)
    x = x + ffn

    return x

# ---------- Build model (token embedding layer saved for weight tying) ----------
inputs = layers.Input(shape=(seq_len,), dtype=tf.int32)

# Keep a reference to the embedding layer so we can tie weights later
token_embedding_layer = layers.Embedding(
    input_dim=vocab_size,
    output_dim=d_model,
    embeddings_initializer=initializer,
    mask_zero=False,  # don't use mask with causal LM
    name="token_embedding"
)

token_emb = token_embedding_layer(inputs)


x = token_emb
x = layers.Dropout(dropout)(x)

for i in range(n_layers):
    x = transformer_block(x, d_model, heads, dropout)

x = layers.RMSNormalization(epsilon=ln_eps)(x)
emb_weights = token_embedding_layer.embeddings  # variable

def lm_head(x):
    return tf.matmul(x, emb_weights, transpose_b=True)

output = layers.Lambda(lm_head, name="logits")(x)

model = tf.keras.Model(inputs, output)

# ---------- Optimizer: use GPT-like defaults ----------
opt = optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.1,
    beta_1=0.9,
    beta_2=0.95,
    epsilon=1e-8,
    clipnorm=1.0
)

model.compile(
    optimizer=opt,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

class PeriodicCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self, save_every_n_steps=400, filepath_prefix='checkpoint_step'):
        super().__init__()
        self.save_every_n_steps = save_every_n_steps
        self.filepath_prefix = filepath_prefix
        self.step_count = 0
    
    def on_batch_end(self, batch, logs=None):
        self.step_count += 1
        
        if self.step_count % self.save_every_n_steps == 0:
            filepath = f"{self.filepath_prefix}_{self.step_count}.weights.h5"
            self.model.save_weights(filepath, overwrite=True)
            print(f"\nSaved weights to {filepath}")
model.load_weights("/content/drive/MyDrive/fine_tuned_rope.weights.h5", skip_mismatch=True)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=2, 
    verbose=1,
    callbacks=[PeriodicCheckpoint(save_every_n_steps=5000)]
)
